# Подготовка данных

Очистка, разбиение train/val/test и подвыборка IACIS 2022. Вся логика разбита на файлы в `src/`

Что делаем:
1. удаляем строки без цены, цену логарифмируем
2. удаляем дубликаты по `description`, чтобы одинаковые тексты не попали в train и test
3. фиксированное разбиение 80/10/10
4. подвыборка IACIS 2022: для сравнения со статьей

In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.config import DATA_PROCESSED, TEXT_COL, POINTS_COL, PRICE_COL, LOG_PRICE_COL
from src.data import load_raw, clean, iacis_categories
from src.split import make_splits, save_splits, load_splits, load_iacis_splits

pd.set_option("display.max_colwidth", 120)

## 1. Исходные данные

In [3]:
raw = load_raw()
print(raw.shape)
raw.head(3)

(129971, 13)


,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,Italy,"Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripen...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,Portugal,"This is ripe and fruity, a wine that is smooth while still structured. Firm tannins are filled out with juicy red be...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,US,"Tart and snappy, the flavors of lime flesh and rind dominate. Some green pineapple pokes through, with crisp acidity...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm


In [4]:
# пропуски по колонкам
pd.DataFrame({"пропусков": raw.isna().sum(), "доля": raw.isna().mean().round(3)})

,пропусков,доля
country,63,0.000
description,0,0.000
designation,37465,0.288
points,0,0.000
price,8996,0.069
province,63,0.000
region_1,21247,0.163
region_2,79460,0.611
taster_name,26244,0.202
taster_twitter_handle,31213,0.240


In [5]:
print("дубликатов по description:", raw[TEXT_COL].duplicated().sum())
print("полных дубликатов строк:", raw.duplicated().sum())
print("диапазон points:", raw[POINTS_COL].min(), "–", raw[POINTS_COL].max())

дубликатов по description: 10016
полных дубликатов строк: 9983
диапазон points: 80 – 100


## 2. Очистка

In [6]:
step1 = raw.dropna(subset=[TEXT_COL, POINTS_COL, PRICE_COL])
step2 = step1.drop_duplicates(subset=[TEXT_COL])
funnel = pd.DataFrame({
    "шаг": ["исходно", "без пропусков description/points/price", "без дубликатов description"],
    "строк": [len(raw), len(step1), len(step2)],
})
funnel["удалено"] = (-funnel["строк"].diff()).fillna(0).astype(int)
funnel

,шаг,строк,удалено
0,исходно,129971,0
1,без пропусков description/points/price,120975,8996
2,без дубликатов description,111567,9408


In [7]:
df = clean(raw)
assert len(df) == len(step2)
assert df[TEXT_COL].is_unique and df[PRICE_COL].notna().all()
print(df.shape)
df[[POINTS_COL, PRICE_COL, LOG_PRICE_COL]].describe().round(3)

(111567, 14)


,points,price,log_price
count,111567.000,111567.000,111567.000
mean,88.414,35.621,3.317
std,3.099,42.107,0.659
min,80.000,4.000,1.386
25%,86.000,17.000,2.833
50%,88.000,25.000,3.219
75%,91.000,42.000,3.738
max,100.000,3300.000,8.102


In [8]:
dup_titles = df[df["title"].duplicated(keep=False)].sort_values("title")
print("строк с повторяющимся title:", df["title"].duplicated().sum())
dup_titles[["title", POINTS_COL, PRICE_COL, TEXT_COL]].head(4)

строк с повторяющимся title: 955


,title,points,price,description
49644,1+1=3 NV Brut Sparkling (Cava),87,16.0,"Spiced apple and toast aromas are clean and direct. On the palate, it's malic and citric, with lime and white grapef..."
81957,1+1=3 NV Brut Sparkling (Cava),87,16.0,"Dusty notes of citrus fruits are mild but friendly. This Cava is zesty, fresh and simple as a whole, with lime, clov..."
33413,A.R. Lenoble NV Dosage Zéro Brut Nature (Champagne),89,49.0,"Bone dry and crisp, this is also has fruitiness from the ripeness of the grapes, giving impressively rich apple and ..."
34492,A.R. Lenoble NV Dosage Zéro Brut Nature (Champagne),90,49.0,This tight crisp wine delivers citrus and sliced apple fruits. There is a brisk mousse that gives the wine quite a b...


## 3. Разбиение train / val / test

In [9]:
train, val, test = make_splits(df)
save_splits(train, val, test)
train, val, test = load_splits()
parts = {"train": train, "val": val, "test": test}

pd.DataFrame({
    n: {
        "строк": len(p),
        "доля": round(len(p) / len(df), 3),
        "points mean": round(p[POINTS_COL].mean(), 3),
        "points std": round(p[POINTS_COL].std(), 3),
        "log_price mean": round(p[LOG_PRICE_COL].mean(), 3),
        "log_price std": round(p[LOG_PRICE_COL].std(), 3),
    }
    for n, p in parts.items()
}).T

,строк,доля,points mean,points std,log_price mean,log_price std
train,89253.0,0.8,88.414,3.099,3.317,0.660
val,11157.0,0.1,88.413,3.098,3.318,0.658
test,11157.0,0.1,88.415,3.100,3.311,0.659


In [10]:
texts = {n: set(p[TEXT_COL]) for n, p in parts.items()}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    print(f"{a} ∩ {b}:", len(texts[a] & texts[b]))
    assert not texts[a] & texts[b]

train ∩ val: 0
train ∩ test: 0
val ∩ test: 0


утечек текста между частями нет

## 4. Подвыборка IACIS 2022 

Есть статья 2022 года с конференции IACIS, где авторы тоже предсказывали оценку вина по этому же датасету Wine Enthusiast. У них R² = 0.593. Но они обучали модель не на всех данных, а на части: 12 стран, 26 сортов, 12 дегустаторов и только вина с указанной ценой.

Нельзя просто взять наш R² на всех данных и поставить рядом с их 0.593: выборки разные, и сравнение будет нечестным. Поэтому мы вырезаем похожую подвыборку, считаем R² на ней и уже его сравниваем с 0.593

Точных списков из статьи нет, поэтому берём самые частые категории: 12 стран, 26 сортов, 12 дегустаторов

In [12]:
cats = iacis_categories(pd.concat(parts.values()))
for col, values in cats.items():
    print(f"{col} ({len(values)}):", ", ".join(values))
    print()

country (12): US, France, Italy, Spain, Portugal, Chile, Argentina, Austria, Australia, Germany, New Zealand, South Africa

variety (26): Pinot Noir, Chardonnay, Cabernet Sauvignon, Red Blend, Bordeaux-style Red Blend, Riesling, Sauvignon Blanc, Syrah, Rosé, Merlot, Zinfandel, Malbec, Sangiovese, Nebbiolo, Portuguese Red, White Blend, Sparkling Blend, Tempranillo, Rhône-style Red Blend, Pinot Gris, Cabernet Franc, Champagne Blend, Grüner Veltliner, Pinot Grigio, Viognier, Portuguese White

taster_name (12): Roger Voss, Michael Schachner, Kerin O’Keefe, Paul Gregutt, Virginie Boone, Matt Kettmann, Joe Czerwinski, Sean P. Sullivan, Anna Lee C. Iijima, Jim Gordon, Anne Krebiehl MW, Lauren Buzzeo



In [13]:
i_train, i_val, i_test = load_iacis_splits(rebuild=True)
iparts = {"train": i_train, "val": i_val, "test": i_test}
pd.DataFrame({
    n: {
        "строк": len(p),
        "доля от основной части": round(len(p) / len(parts[n]), 3),
        "points mean": round(p[POINTS_COL].mean(), 3),
        "points std": round(p[POINTS_COL].std(), 3),
        "log_price mean": round(p[LOG_PRICE_COL].mean(), 3),
    }
    for n, p in iparts.items()
}).T

,строк,доля от основной части,points mean,points std,log_price mean
train,53829.0,0.603,88.714,3.043,3.348
val,6799.0,0.609,88.715,3.039,3.353
test,6612.0,0.593,88.730,3.050,3.349


In [14]:
isub = pd.concat(iparts.values())
print("всего строк:", len(isub))
print({c: isub[c].nunique() for c in cats})
assert isub[PRICE_COL].notna().all()

всего строк: 67240
{'country': 12, 'variety': 26, 'taster_name': 12}


## 5. Сохранённые файлы

In [15]:
pd.DataFrame(
    [(f.name, round(f.stat().st_size / 2**20, 1)) for f in sorted(DATA_PROCESSED.glob("*.parquet"))],
    columns=["файл", "МБ"],
)

,файл,МБ
0,iacis.parquet,12.4
1,test.parquet,2.2
2,train.parquet,17.1
3,val.parquet,2.2
